# 문서 파싱 테스트 (LLM 호출 없이 파싱만)

`app/services/document_parser_service.py`의 PDF/Word(.docx) 파싱 로직만 떼어서 확인하기 위한 노트북입니다. Azure OpenAI 등 LLM 호출은 포함하지 않습니다.

## 사전 준비
1. 아래 패키지가 설치되어 있어야 합니다.

```bash
pip install jupyter ipykernel pypdf python-docx
```
2. 테스트 데이터는 `data/sample/` 폴더의 파일을 사용합니다 (예: `지능망서비스+이용약관_2026.pdf`).

In [ ]:
from io import BytesIO
from pathlib import Path

from docx import Document
from pypdf import PdfReader

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/) 기준 실제 샘플 데이터 폴더
SAMPLES_DIR = Path.cwd().parent / "data" / "sample"

print("SAMPLES_DIR:", SAMPLES_DIR)

## 1. 문서 파싱 함수 (pypdf / python-docx)

`document_parser_service.py`와 동일한 로직입니다.

In [3]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx"}


def parse_pdf(content: bytes) -> str:
    reader = PdfReader(BytesIO(content))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages).strip()


def parse_docx(content: bytes) -> str:
    document = Document(BytesIO(content))
    paragraphs = [p.text for p in document.paragraphs if p.text]
    return "\n".join(paragraphs).strip()


def parse_document(filename: str, content: bytes) -> str:
    extension = Path(filename).suffix.lower()

    if extension == ".pdf":
        return parse_pdf(content)
    if extension == ".docx":
        return parse_docx(content)

    raise ValueError(f"지원하지 않는 파일 형식입니다: {filename} (지원: {SUPPORTED_EXTENSIONS})")

## 2. 단일 파일 파싱 테스트

In [ ]:
# 테스트할 파일명을 data/sample 폴더 기준으로 지정하세요
TEST_FILENAME = "지능망서비스+이용약관_2026.pdf"

test_path = SAMPLES_DIR / TEST_FILENAME
content_bytes = test_path.read_bytes()

parsed_text = parse_document(test_path.name, content_bytes)

print("char_count:", len(parsed_text))
print("--- preview (앞 500자) ---")
print(parsed_text[:500])

## 3. 여러 파일 한 번에 파싱 테스트

`data/sample/` 폴더에 여러 개의 PDF/DOCX를 넣어두고, 파일명 목록만 바꿔가며 한 번에 확인할 때 사용하세요.

In [ ]:
def run_parse(filename: str) -> dict:
    path = SAMPLES_DIR / filename
    content = path.read_bytes()

    try:
        text = parse_document(path.name, content)
        return {"filename": filename, "char_count": len(text), "preview": text[:200]}
    except ValueError as exc:
        return {"filename": filename, "error": str(exc)}


# data/sample 폴더에 넣어둔 파일명들로 채우세요
TEST_FILENAMES = ["지능망서비스+이용약관_2026.pdf"]

for name in TEST_FILENAMES:
    result = run_parse(name)
    print(result)
    print("---")